In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/processed/cases_final.csv")

print(df.shape)
df.head()

(31, 9)


,case_id,file_name,text_full,no_perkara,penggugat,tergugat,amar_putusan,jenis_perkara,pengadilan
0,case_001,putusan_105_pdt.g_2025_pn_sda_20260624161353.pdf,penetapan nomor 105/pdt.g/2025/pn sda esi demi...,105/pdt.g/2025/pnsda,1. sri pudjiastuti bertempat tinggal di jalan....,1. siti rusmala bertempat tinggal di desa boha...,1. mengabulkan pencabutan permohonan perkara ...,Wanprestasi,PN Sidoarjo
1,case_002,putusan_148_pdt.g_2025_pn_sda_20260624161238.pdf,putusan nomor 148/pdt.g/2025/pn sda demi keadi...,148/pdt.g/2025/pnsda,-,les 1. kepala dinas perhubungan kabupaten sido...,nomor 415.4/34/438.5.13/2023nomor 107/lds-sdm...,Wanprestasi,PN Sidoarjo
2,case_003,putusan_149_pdt.g_2017_pn_sda_20260624160009.pdf,direktori putusan mahkamah agung republik indo...,149/pdt.g/2017/pnsda,onesi andry tejokusuma berkedudukan di jl. jem...,h. imam ghozali atau imam ghozali bertempat ti...,dalamkonpensi dalam eksepsi nkamah dalam poko...,Wanprestasi,PN Sidoarjo
3,case_004,putusan_166_pdt.g_2021_pn_sda_20260624160223.pdf,direktori putusan mahkamah agung pdt.i.c.3 put...,166/pdt.g/2021/pnsda,raditya afrisal hidayat berkedudukan di perum ...,rachmad harmoko bertempat tinggal di jalan kar...,1. menyatakan bahwa tergugat yang telah dipan...,Wanprestasi,PN Sidoarjo
4,case_005,putusan_213_pdt.g_2026_pn_sda_20260624160912.pdf,penetapan nomor 213/pdt.g/2026/pn sda demi kea...,213/pdt.g/2026/pnsda,wahjudi santoso tempat dan tanggal lahir sidoa...,avi anthy nurafni tempat dan tanggal lahir ung...,perkara ini setelah membaca penetapan hakim n...,Wanprestasi,PN Sidoarjo


In [3]:
vectorizer = TfidfVectorizer(
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(df["text_full"])

print(tfidf_matrix.shape)

(31, 5000)


In [4]:
def retrieve(query, k=5):
    query_vector = vectorizer.transform([query])
    similarity = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_index = similarity.argsort()[-k:][::-1]
    
    result = df.iloc[top_index].copy()
    result["similarity_score"] = similarity[top_index]
    
    return result[[
        "case_id",
        "no_perkara",
        "penggugat",
        "tergugat",
        "similarity_score",
        "amar_putusan"
    ]]

In [5]:
query = """
tergugat tidak memenuhi perjanjian dan melakukan wanprestasi
sehingga penggugat mengalami kerugian
"""

retrieve(query, k=5)

,case_id,no_perkara,penggugat,tergugat,similarity_score,amar_putusan
22,case_023,390/pdt.g/2023/pnsda,rudik irwanto lahir di sidoarjo tanggal 10 mei...,les 1. sri hartini jenis kelamin perempuan umu...,0.322963,1. menyatakan tergugat i dan tergugat ll yang...
23,case_024,408/pdt.g/2026/pnsda,ariesca dwi aptasari sh m.kn nik 3515084404850...,il/1 rt.030 rw.006 kel. sidokumpul kec. sidoar...,0.242264,perkara perdata pada tingkat pertama telah me...
24,case_025,-,-,ikamah pengawasan dan pelayanan tipe madya pab...,0.232176,perkara ini berkenan memutuskan sebagai berik...
21,case_022,376/pdt.g/2025/pnsda,tutut indriana wijaya beralamat di jalan brawi...,indah melisa bertempat tinggal di perumahan ta...,0.224695,dengan menjatuhkan putusan yang amarnya sebag...
14,case_015,300/pdt.g/2022/pnsda,-,h. iswandi bertempat tinggal di desa kalidawir...,0.220153,inkamah dalam eksepsi dan kabur dari 39 putus...
